In [ ]:
"""
最小 LangGraph agent —— 直接用 StateGraph 拼出来。

图结构：
    START
      │
      ▼
    ┌────────┐  有 tool_calls  ┌────────┐
    │ agent  │ ───────────────▶│ tools  │
    │  (LLM) │                 │(ToolNode)│
    └────────┘ ◀───────────────└────────┘
      │  无 tool_calls            │
      ▼                           │
     END                          │
      ▲───────────────────────────┘
                  循环

每次循环都会把新产生的 message 完整打印，便于理解整条链路。
"""
import os
import json
from typing import Literal

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

from src.core.tools.weather import get_weather  # 复用项目里现有的 tool

load_dotenv()

MODEL = 'qwen3.6-flash'
API_KEY = os.getenv('LEARN_AGENT_LLM_API_KEY')
BASE_URL = os.getenv('LEARN_AGENT_LLM_BASE_URL')
# Response API 需要模型提供方原生支持；当前 dashscope 网关不支持，可设 false 回退。
USE_RESPONSES_API = os.getenv('LEARN_AGENT_USE_RESPONSES_API', 'true').lower() == 'true'
SYSTEM_PROMPT = "你是一个天气查询助手。当用户询问某城市天气时，必须调用 get_weather 工具获取数据，再以简洁中文回复用户。"

# —— LLM ——
llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.7,
    use_responses_api=USE_RESPONSES_API,
)
# 把 tool 绑到 LLM 上：之后 llm.invoke(messages) 就会产出带 tool_calls 的 AIMessage
llm_with_tools = llm.bind_tools([get_weather])
print(f'[config] model={MODEL}  use_responses_api={USE_RESPONSES_API}')
print(f'[config] tools={[t.name for t in [get_weather]]}')


# —— 节点 1：agent —— 调 LLM，返回新的 AIMessage
def agent_node(state: MessagesState) -> dict:
    msgs = state["messages"]
    response = llm_with_tools.invoke(msgs)
    # 返回 dict 形式的增量 state；MessagesState 会自动追加
    return {"messages": [response]}


# —— 节点 2：tools —— langgraph 自带的 ToolNode，按 AIMessage.tool_calls 调工具
tools_node = ToolNode([get_weather])


# —— 条件边：agent 之后看最后一条 message 有没有 tool_calls ——
def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return "__end__"


# —— 拼图 ——
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tools_node)

graph_builder.add_edge(START, "agent")                          # START -> agent
graph_builder.add_conditional_edges("agent", should_continue)   # agent -> tools / END
graph_builder.add_edge("tools", "agent")                        # tools -> agent (再问一次 LLM)

graph = graph_builder.compile()
print(f'[config] graph_nodes={list(graph.get_graph().nodes.keys())}')


# —— 整条 message 完整 dump，原始内容不修改、不筛选字段 ——
def dump_message(msg, tag: str = '') -> None:
    role = getattr(msg, "type", msg.__class__.__name__)
    print(f'\n========== [{role}]{(" " + tag) if tag else ""} ==========')
    print(json.dumps(msg.model_dump(), indent=2, ensure_ascii=False))
    print('=' * 35)


def run_one_turn(user_input: str, history: list) -> list:
    """跑一轮：把人类输入 + 每个节点的输出完整打出来。"""
    print(f'\n>>> HUMAN INPUT: {user_input!r}')
    history.append({"role": "user", "content": user_input})

    # stream_mode='values' 会在每个节点执行后产出完整 state
    last_msg_count = 0
    final_state = None
    for event in graph.stream({"messages": history}, stream_mode="values"):
        msgs = event.get("messages", [])
        for msg in msgs[last_msg_count:]:
            tag = 'after agent node' if getattr(msg, 'type', '') == 'ai' else \
                  'after tools node' if getattr(msg, 'type', '') == 'tool' else ''
            dump_message(msg, tag)
        last_msg_count = len(msgs)
        final_state = event

    if final_state:
        history = final_state["messages"]
    return history


# —— 简单 agent loop ——
history = []
print('\n[agent loop ready] 输入城市名问天气，输入 q 退出。')
while True:
    try:
        user_input = input('\n>> ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n[bye]')
        break
    if not user_input:
        continue
    if user_input.lower() in ('q', 'quit', 'exit'):
        print('[bye]')
        break
    try:
        history = run_one_turn(user_input, history)
    except Exception as e:
        print(f'\n[error] {type(e).__name__}: {e}')
        # 出错时撤回刚 append 的 user 输入，避免污染 history
        if history and history[-1].get('role') == 'user' and history[-1].get('content') == user_input:
            history.pop()


In [7]:
"""
最小 LangGraph agent —— Anthropic 后端。

图结构和 cell 1 完全一致：
    START -> agent -> (有 tool_calls ? tools : END)
                  ^      |
                  └──────┘

唯一区别：把 ChatOpenAI 换成 ChatAnthropic。
"""
import os
import json
from typing import Literal

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

from src.core.tools.weather import get_weather  # 复用项目里现有的 tool

load_dotenv()

MODEL = 'deepseek-v4-flash'
API_KEY = os.getenv('LEARN_AGENT_LLM_API_KEY')
BASE_URL = 'https://dashscope.aliyuncs.com/apps/anthropic'  # 兼容 Anthropic 协议的 dashscope 网关
SYSTEM_PROMPT = "你是一个天气查询助手。当用户询问某城市天气时，必须调用 get_weather 工具获取数据，再以简洁中文回复用户。"

# —— LLM ——
# Anthropic 不像 OpenAI 那样有 use_responses_api 开关，走的就是 /v1/messages。
# max_tokens 是 Anthropic 必填字段（无默认值）。
llm = ChatAnthropic(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.7,
    max_tokens=4096,
)
llm_with_tools = llm.bind_tools([get_weather])
print(f'[config] model={MODEL}  base_url={BASE_URL}')


# —— 节点 1：agent ——
def agent_node(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


# —— 节点 2：tools ——
tools_node = ToolNode([get_weather])


# —— 条件边 ——
def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else "__end__"


# —— 拼图 ——
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tools_node)
graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges("agent", should_continue)
graph_builder.add_edge("tools", "agent")
graph = graph_builder.compile()
print(f'[config] graph_nodes={list(graph.get_graph().nodes.keys())}')


# —— 整条 message 完整 dump，原始内容不修改、不筛选字段 ——
def dump_message(msg, tag: str = '') -> None:
    role = getattr(msg, "type", msg.__class__.__name__)
    print(f'\n========== [{role}]{(" " + tag) if tag else ""} ==========')
    print(json.dumps(msg.model_dump(), indent=2, ensure_ascii=False))
    print('=' * 35)


def run_one_turn(user_input: str, history: list) -> list:
    """跑一轮：把人类输入 + 每个节点的输出完整打出来。"""
    print(f'\n>>> HUMAN INPUT: {user_input!r}')
    history.append({"role": "user", "content": user_input})

    last_msg_count = 0
    final_state = None
    for event in graph.stream({"messages": history}, stream_mode="values"):
        msgs = event.get("messages", [])
        for msg in msgs[last_msg_count:]:
            tag = 'after agent node' if getattr(msg, 'type', '') == 'ai' else \
                  'after tools node' if getattr(msg, 'type', '') == 'tool' else ''
            dump_message(msg, tag)
        last_msg_count = len(msgs)
        final_state = event

    if final_state:
        history = final_state["messages"]
    return history


# —— 简单 agent loop ——
history = []
print('\n[agent loop ready] 输入城市名问天气，输入 q 退出。')
while True:
    try:
        user_input = input('\n>> ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n[bye]')
        break
    if not user_input:
        continue
    if user_input.lower() in ('q', 'quit', 'exit'):
        print('[bye]')
        break
    try:
        history = run_one_turn(user_input, history)
    except Exception as e:
        print(f'\n[error] {type(e).__name__}: {e}')
        if history and history[-1].get('role') == 'user' and history[-1].get('content') == user_input:
            history.pop()


[config] model=deepseek-v4-flash  base_url=https://dashscope.aliyuncs.com/apps/anthropic
[config] graph_nodes=['__start__', 'agent', 'tools', '__end__']

[agent loop ready] 输入城市名问天气，输入 q 退出。

>>> HUMAN INPUT: '北京'

========== [human] ==========
{
  "content": "北京",
  "additional_kwargs": {},
  "response_metadata": {},
  "type": "human",
  "name": null,
  "id": "43b1cd44-ced4-4226-902c-7f27be377a3f"
}

========== [ai] after agent node ==========
{
  "content": [
    {
      "signature": "",
      "thinking": "用户询问“北京”，可能是想了解北京的天气信息。我可以使用 get_weather 工具来查询北京的当前天气。\n\n让我调用工具获取北京的天气信息。",
      "type": "thinking"
    },
    {
      "id": "toolu_72a4d6fdaba14789a2c2cdb1",
      "input": {
        "city": "北京"
      },
      "name": "get_weather",
      "type": "tool_use"
    }
  ],
  "additional_kwargs": {},
  "response_metadata": {
    "id": "msg_2cb88274-f868-433c-b12c-a6f95eea7a62",
    "container": null,
    "model": "deepseek-v4-flash",
    "stop_details": null,
    "stop_reason": "tool